# Comfort standards defined by their data

## Introduction

Since accim 1.2.0, a comfort standard whose comfort temperature is a straight line of an outdoor metric can be **defined by its data**, in a YAML file, and registered for the rest of the Python session with a number of 100 or more. `AddAccis`, `modify_accis`, `print_setpoint_ems_code`, `compute_setpoints`, `verify_accim_simulation`, the parametric module and the graphical user interface then take it as a built-in comfort standard (section 4.2.5 of the detailed use of accim).

This notebook uses the YAML file of **JPN Aqilah 2025** (comfort standard 101), which the sample files of accim include: the adaptive model of Aqilah, Rijal, Yoshida & Nicol (2025, *Energy & Buildings* 335, 115469) for 64 Japanese dwellings in free-running mode,

$$T_c = 0.38 \cdot T_{rm} + 16.9$$

where $T_{rm}$ is the running mean outdoor temperature with $\alpha = 0.8$ (their equations 8 and 7), applicable from 5 to 30 °C, with comfort bands of ±1.5 K (90 %) and ±2.0 K (80 %). We will:

1. read and register the YAML file, and see why the second sample file (Qatar) is rejected;
2. print the EMS code of its setpoints, with the evidence of the comfort model;
3. compare its setpoints with those of JPN Rijal (comfort standard 3, $T_c = 0.48 \cdot T_{pma} + 14.4$, with the prevailing mean outdoor temperature) in Seville, without EnergyPlus;
4. add the ACCIS with both comfort standards to a model, simulate it, verify the simulation and compare the results.

## 1. Setup and Imports

The folder of this notebook has the model (`TestModel.idf`, two zones, EnergyPlus 25.1), the weather file of Seville (`Seville.epw`) and copies of the two YAML files of `accim/sample_files/comfort_standards`.

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import pandas as pd

from accim.run import run_ep
from accim.sim import accis, load_comfort_standards
from accim.sim.preview import compute_setpoints
from accim.sim.standards import registered_comfort_standards, unregister_comfort_standard
from accim.utils import get_setpoint_ems_code
from accim.utils.verification import verify_accim_simulation

plt.rcParams['figure.figsize'] = [14, 5]

idf_filename = 'TestModel.idf'
epw_filename = 'Seville.epw'
for filename in (idf_filename, epw_filename, 'jpn_aqilah2025_fr.yaml', 'qat_indraganti2018_hvac.yaml'):
    print(f"{filename}: {'found' if os.path.exists(filename) else 'NOT FOUND'}")

## 2. The YAML File

Each key of the file is a datum of the comfort model: its number and names (`label` is the name in the names of the output IDFs), its source (`citation`), its outdoor metric (one of those of the custom model: `pmot`, `rmot`, `rmot_alpha`...), its comfort temperature and applicability limits, the offsets of the setpoints of each category, its comfort modes, its static setpoints (which comfort modes 0 and 1 need; the paper gives none, so 20 and 28 °C are a choice of the user) and its **evidence**, the statistics of the comfort model, which cannot be empty: the reliability of a model depends on them.

In [ ]:
with open('jpn_aqilah2025_fr.yaml', encoding='utf-8') as file:
    print(file.read())

## 3. Registering the Comfort Standard

`load_comfort_standards` reads a YAML file (or the YAML files of a folder), checks it and registers its comfort standards for the rest of the session (`register=False` only reads them). It returns their definitions.

In [ ]:
[aqilah] = load_comfort_standards('jpn_aqilah2025_fr.yaml')

print(aqilah.standard.title)
print(f'Outdoor metric: {aqilah.outdoor_metric}, alpha {aqilah.alpha}')
print(f'Comfort temperature: {aqilah.regimes[0].slope} * metric + {aqilah.regimes[0].intercept}')
print(f'Offsets of the categories (cooling, heating): {aqilah.category_offsets}')
print('Comfort modes:')
for mode, description in aqilah.standard.comfort_modes.items():
    print(f'    {mode}: {description}')
print(f'Registered comfort standards: {list(registered_comfort_standards())}')

The second sample file has the comfort temperature and the evidence of the offices with HVAC of Qatar (Indraganti & Boussaa, 2018), an almost flat line ($R^2$ = 0.015) whose weakness only the evidence shows. Its sources give neither its outdoor metric, nor its comfort band, nor its applicability limits, so these keys are empty, and `load_comfort_standards` rejects the file with a message that names it and the keys:

In [ ]:
try:
    load_comfort_standards('qat_indraganti2018_hvac.yaml')
except ValueError as error:
    print(f'ValueError: {error}')

## 4. The EMS Code of the Setpoints

`get_setpoint_ems_code` (and `print_setpoint_ems_code`) gives the EMS code with which the ACCIS calculates the setpoints of a comfort standard, category and comfort mode, as in the output IDFs. For a comfort standard defined by its data, the comments at the top give its citation, its outdoor metric, its static setpoints and its evidence. Its outdoor metric (`rmot_alpha`) is calculated by the EMS (`Tout_metric`), with the programs of the outdoor metric of the custom model, so the code has them too; we print the comments, the comfort temperature and the setpoints of category 80 and comfort mode 1 (the adaptive setpoints within the applicability limits, otherwise the static setpoints):

In [ ]:
code = get_setpoint_ems_code(comfort_standard=101, category=80, comfort_mode=1)
lines = code.split('\n')
# The comments at the top
print('\n'.join(lines[:lines.index('')]))
# The comfort temperature, and the setpoints of the comfort mode
for title in ('! --- SetComfTemp', '! --- SetAST_CS101_0'):
    start = next(number for number, line in enumerate(lines) if line.startswith(title))
    print()
    print('\n'.join(lines[start:lines.index('', start)]))

## 5. The Setpoints without EnergyPlus

`compute_setpoints` runs that EMS code with the Erl interpreter of accim, for the days of a weather file, without EnergyPlus. We compare JPN Aqilah 2025 with JPN Rijal (comfort standard 3), with category 80 and comfort mode 3 (out of the applicability limits, the setpoints of the limits). `Tout_metric` is the outdoor metric of each one: the running mean with $\alpha = 0.8$ for JPN Aqilah 2025, and the prevailing mean outdoor temperature of EnergyPlus (PMOT) for JPN Rijal.

In [ ]:
setpoints = {
    'JPN Aqilah 2025 (101)': compute_setpoints(epw_filename, 101, 80, 3),
    'JPN Rijal (3)': compute_setpoints(epw_filename, 3, 80, 3),
}
colors = {'JPN Aqilah 2025 (101)': 'tab:blue', 'JPN Rijal (3)': 'tab:orange'}

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={'width_ratios': [2, 1]})
for name, df in setpoints.items():
    daily = df.set_index('datetime')[['ACST', 'AHST']].resample('D').mean()
    axes[0].plot(daily.index, daily['ACST'], color=colors[name], label=f'Cooling setpoint, {name}')
    axes[0].plot(daily.index, daily['AHST'], color=colors[name], linestyle='--', label=f'Heating setpoint, {name}')
    axes[1].scatter(df['Tout_metric'], df['ComfTemp'], s=2, color=colors[name], label=name)
axes[0].set_ylabel('Setpoint (°C), daily mean')
axes[0].set_title('Adaptive setpoints in Seville, category 80, comfort mode 3')
axes[0].legend(fontsize=8)
axes[1].set_xlabel('Outdoor metric (°C)')
axes[1].set_ylabel('Comfort temperature (°C)')
axes[1].set_title('Comfort temperature')
axes[1].legend(markerscale=5)
plt.tight_layout()
plt.show()

pd.DataFrame({name: {'Mean cooling setpoint (°C)': df['ACST'].mean(),
                     'Mean heating setpoint (°C)': df['AHST'].mean(),
                     'Range of the outdoor metric (°C)': f"{df['Tout_metric'].min():.1f} to {df['Tout_metric'].max():.1f}",
                     'Hours within the applicability limits (%)': 100 * df['applicable'].mean()}
              for name, df in setpoints.items()}).round(2)

## 6. Adding the ACCIS

The comfort standard is now one of those of `AddAccis`: we add the ACCIS with both comfort standards to the model, in mixed mode (`vrf_mm`: the windows open when the zone is comfortable, and the VRF system works otherwise), with category 80 and comfort mode 3. The weather file of the simulations (`epw`) begins the outdoor metric of JPN Aqilah 2025, which the EMS calculates, with its values. Each output IDF only has the EMS code of its comfort standard: that of JPN Rijal is the same as without the registry.

In [ ]:
accis.AddAccis(
    idfs=[idf_filename],
    script_type='vrf_mm',
    supply_air_temp_input_method='temperature difference',
    output_type='standard',
    output_frequencies=['hourly'],
    keep_existing_outputs=False,
    energyplus_version='auto',
    setpoint_type='temperature',
    comfort_standard=[3, 101],
    category=[80],
    comfort_mode=[3],
    hvac_mode=[2],
    ventilation_control=[0],
    epw=epw_filename,
    confirm=True,
    verbose=False,
)
output_idfs = sorted(name for name in os.listdir() if name.startswith('TestModel[') and name.endswith('.idf'))
for name in output_idfs:
    print(name)

## 7. Simulations

`run_ep` simulates the output IDFs of the folder with the weather file (about a minute each).

In [ ]:
run_ep(only_accim_idfs=True, confirm=True, processes=2, energyplus_version='auto')

## 8. Verifying the Simulation

`verify_accim_simulation` checks the outputs of a simulation; with the weather file, its check 4 compares the comfort temperature, the setpoints and the outdoor metric of each zone with those of `compute_setpoints`. The comfort standard must be registered, as here.

In [ ]:
aqilah_idf = next(name for name in output_idfs if 'Aqilah' in name)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    mismatches = verify_accim_simulation(f"{aqilah_idf[:-4]}[{epw_filename[:-4]}.eso", aqilah_idf, epw=epw_filename)
print(f'Mismatches: {len(mismatches)}')

## 9. Results

We read the hourly results of both simulations, and compare the setpoints and the operative temperature of the first zone in a summer week, and, for the whole year, the HVAC electricity, the hours with the windows open (ventilation hours) and the occupied hours within the setpoints of both zones.

In [ ]:
results = {}
for name in output_idfs:
    standard = 'JPN Aqilah 2025 (101)' if 'Aqilah' in name else 'JPN Rijal (3)'
    df = pd.read_csv(f"{name[:-4]}[{epw_filename[:-4]}.csv")
    results[standard] = df.iloc[-8760:].reset_index(drop=True)


def column(df, *texts):
    # The (first) column whose name has all the texts
    return df[next(c for c in df.columns if all(text in c for text in texts))]


week = slice(24 * 195, 24 * 202)
fig, ax = plt.subplots(figsize=(15, 5))
for name, df in results.items():
    ax.plot(column(df, 'Adaptive Cooling Setpoint Temperature_Block1_Zone1').iloc[week].values, color=colors[name],
            linestyle='--', label=f'Cooling setpoint, {name}')
    ax.plot(column(df, 'BLOCK1:ZONE1:Zone Operative Temperature').iloc[week].values, color=colors[name],
            label=f'Operative temperature, {name}')
ax.set_xlabel('Hour of the week (15 to 21 July)')
ax.set_ylabel('Temperature (°C)')
ax.set_title('Block1:Zone1 in a summer week')
ax.legend(fontsize=8)
plt.show()

summary = {}
for name, df in results.items():
    occupied = sum(column(df, f'Occupied Hours_Block1_{zone}').sum() for zone in ('Zone1', 'Zone2'))
    comfortable = sum(column(df, f'Occupied Comfortable Hours_No Applicability_Block1_{zone}').sum()
                      for zone in ('Zone1', 'Zone2'))
    summary[name] = {
        'HVAC electricity (kWh)': column(df, 'Electricity:HVAC').sum() / 3.6e6,
        'Ventilation hours, both zones': sum(column(df, f'Ventilation Hours_Block1_{zone}').sum()
                                             for zone in ('Zone1', 'Zone2')),
        'Occupied hours within the setpoints (%)': 100 * comfortable / occupied,
    }
pd.DataFrame(summary).T.round(1)

## 10. Conclusion

1. A comfort standard defined by its data in a YAML file becomes, once registered, one more comfort standard of accim, with a number of 100 or more: its EMS code is written as that of the built-in comfort standards, whose output IDFs do not change.
2. `print_setpoint_ems_code` shows its source and its evidence with its EMS code, so the statistics of the comfort model travel with it; a file without the data that the EMS needs is rejected.
3. JPN Aqilah 2025, with its running mean outdoor temperature and its comfort band of category 80 (±2.0 K, narrower than the ±3.5 K of JPN Rijal in accim, taken from ASHRAE 55), gives in Seville setpoints that differ from those of JPN Rijal, the earlier Japanese model of accim: here, the windows of mixed mode open fewer hours, and the VRF systems use more electricity.

The registry is that of the Python session: register the comfort standards again in each session, before making, modifying or checking the models that have them.

## 11. Cleanup

Finally, we remove the output IDFs and the results of the simulations, and the comfort standard from the registry. **Warning:** This will permanently delete the files generated in this notebook.

In [ ]:
removed = [name for name in os.listdir() if name.startswith('TestModel[')]
for name in removed:
    os.remove(name)
print(f'Removed {len(removed)} files: the output IDFs and the results of their simulations')
unregister_comfort_standard(101)
print(f'Registered comfort standards: {list(registered_comfort_standards())}')